# Text Side Ablation

Retrains all models with `use_text=True` — a frozen MiniLM embedding of the root tweet concatenated at readout.
Every model uses the same hyperparameters as its structure-only run, same split, same 5 seeds.

Runs: 5 models × 2 datasets × 5 seeds = **50 new runs**.

In [ ]:
import sys
sys.path.insert(0, "..")

import json, os, glob, warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch

from data.dataset import CascadeDataset
from models.gcn import GCNClassifier
from models.gat import GATClassifier
from models.bigcn import BiGCNClassifier
from models.gps import GPSClassifier
from models.cascade_gps import CascadeGPSClassifier
from training.trainer import run_experiment

DATA_ROOT = "../Twitter15_Twitter16"
RESULTS_DIR = "../results"
SEEDS = [0, 1, 2, 3, 4]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
ds15 = CascadeDataset(root=DATA_ROOT, name="twitter15")
ds16 = CascadeDataset(root=DATA_ROOT, name="twitter16")
IN_CHANNELS = ds15[0].x.shape[1]
EDGE_DIM = ds15[0].edge_attr.shape[1]
assert ds15[0].root_text.shape == (1, 384) and ds16[0].root_text.shape == (1, 384)
print(f"Twitter15: {len(ds15)}  Twitter16: {len(ds16)}  in_channels={IN_CHANNELS}  edge_dim={EDGE_DIM}")

## Training runs

One cell per model family so each can be run independently.

In [ ]:
GCN_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, dropout=0.1, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"GCN+text — {ds_name}")
    run_experiment(
        GCNClassifier, GCN_KWARGS, ds, SEEDS,
        epochs=200, warmup_ratio=0.1, device=DEVICE,
        results_dir=RESULTS_DIR, model_name="gcn", dataset_name=ds_name,
    )

In [ ]:
GAT_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, heads=4, dropout=0.1, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"GAT+text — {ds_name}")
    run_experiment(
        GATClassifier, GAT_KWARGS, ds, SEEDS,
        epochs=200, warmup_ratio=0.1, device=DEVICE,
        results_dir=RESULTS_DIR, model_name="gat", dataset_name=ds_name,
    )

In [ ]:
BIGCN_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, dropout=0.1, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"BiGCN+text — {ds_name}")
    run_experiment(
        BiGCNClassifier, BIGCN_KWARGS, ds, SEEDS,
        epochs=200, warmup_ratio=0.1, device=DEVICE,
        results_dir=RESULTS_DIR, model_name="bigcn", dataset_name=ds_name,
    )

In [ ]:
GPS_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=4, heads=4,
                  dropout=0.1, edge_dim=EDGE_DIM, readout="4way", use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"GraphGPS+text — {ds_name}")
    run_experiment(
        GPSClassifier, GPS_KWARGS, ds, SEEDS,
        epochs=200, lr=1e-3, weight_decay=0.05, patience=40,
        warmup_ratio=0.1, lap_pe_sign_flip=True, max_nodes_per_batch=8192,
        device=DEVICE,
        results_dir=RESULTS_DIR, model_name="gps", dataset_name=ds_name,
    )

In [ ]:
CASCADE_GPS_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, heads=4,
                          dropout=0.1, edge_dim=EDGE_DIM, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"CascadeGPS+text — {ds_name}")
    run_experiment(
        CascadeGPSClassifier, CASCADE_GPS_KWARGS, ds, SEEDS,
        epochs=200, lr=1e-3, weight_decay=0.05, patience=40,
        warmup_ratio=0.1, lap_pe_sign_flip=True, max_nodes_per_batch=2048,
        device=DEVICE,
        results_dir=RESULTS_DIR, model_name="cascade_gps", dataset_name=ds_name,
    )

## Results table

Compares text-on vs text-off by loading both sets of per-seed JSONs from `results/`.

In [ ]:
import numpy as np
from analysis.stats import load_run_files, group_runs, summarise

runs = load_run_files(RESULTS_DIR)
groups = group_runs(runs)

ALL_MODELS = [
    ("gcn",         "mean"),
    ("gat",         "mean"),
    ("bigcn",       "mean"),
    ("gps",         "4way"),
    ("cascade_gps", "mean"),
]

print(f"{'Model':<14} {'Dataset':<12} {'text=off':>12} {'text=on':>12} {'Δ F1':>8}")
print("-" * 62)
for model, readout in ALL_MODELS:
    for dataset in ("twitter15", "twitter16"):
        off = groups.get((model, dataset, False, readout))
        on  = groups.get((model, dataset, True,  readout))
        s_off = f"{summarise(off)['mean']:.3f} ±{summarise(off)['std']:.3f}" if off else "—"
        s_on  = f"{summarise(on)['mean']:.3f} ±{summarise(on)['std']:.3f}"  if on  else "—"
        if off and on:
            delta = summarise(on)["mean"] - summarise(off)["mean"]
            s_delta = f"{delta:+.3f}"
        else:
            s_delta = "—"
        print(f"{model:<14} {dataset:<12} {s_off:>12} {s_on:>12} {s_delta:>8}")